# Gridalytics - Model Comparison

Compare LightGBM, XGBoost, and Ensemble models with walk-forward cross-validation.

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import date

from src.data.db.session import get_session
from src.features.pipeline import FeaturePipeline
from src.models.lightgbm_model import LightGBMForecaster
from src.models.xgboost_model import XGBoostForecaster
from src.evaluation.cross_validation import walk_forward_cv
from src.evaluation.metrics import compute_all_metrics, compute_seasonal_metrics, print_metrics_report

## 1. Build Features

In [ ]:
with get_session() as session:
    pipeline = FeaturePipeline('hourly', session)
    df = pipeline.build(date(2021, 1, 1), date(2026, 3, 28))

target = 'delhi_mw'
features = [c for c in pipeline.get_feature_names(df) if c not in ['brpl_mw','bypl_mw','ndpl_mw','ndmc_mw','mes_mw']]
df[features] = df[features].fillna(method='ffill').fillna(0)
print(f'Data: {len(df):,} rows, {len(features)} features')

## 2. Walk-Forward CV: LightGBM

In [ ]:
lgb_results = walk_forward_cv(LightGBMForecaster, df, target, features, 'hourly')
lgb_results[['fold', 'mape', 'rmse', 'mae', 'r2']].round(4)

## 3. Walk-Forward CV: XGBoost

In [ ]:
xgb_results = walk_forward_cv(XGBoostForecaster, df, target, features, 'hourly')
xgb_results[['fold', 'mape', 'rmse', 'mae', 'r2']].round(4)

## 4. Comparison Chart

In [ ]:
comparison = pd.DataFrame({
    'Fold': range(len(lgb_results)),
    'LightGBM MAPE': lgb_results['mape'].values,
    'XGBoost MAPE': xgb_results['mape'].values,
})

fig = go.Figure()
fig.add_trace(go.Bar(x=comparison['Fold'], y=comparison['LightGBM MAPE'], name='LightGBM', marker_color='#3b82f6'))
fig.add_trace(go.Bar(x=comparison['Fold'], y=comparison['XGBoost MAPE'], name='XGBoost', marker_color='#f59e0b'))
fig.update_layout(title='MAPE by Fold (Walk-Forward CV)', xaxis_title='Fold', yaxis_title='MAPE %',
                  barmode='group', template='plotly_dark')
fig.show()

## 5. Summary

In [ ]:
print('LightGBM:')
print(f"  Mean MAPE: {lgb_results['mape'].mean():.2f}% +/- {lgb_results['mape'].std():.2f}%")
print(f"  Mean RMSE: {lgb_results['rmse'].mean():.1f}")
print(f"  Mean R2:   {lgb_results['r2'].mean():.4f}")
print()
print('XGBoost:')
print(f"  Mean MAPE: {xgb_results['mape'].mean():.2f}% +/- {xgb_results['mape'].std():.2f}%")
print(f"  Mean RMSE: {xgb_results['rmse'].mean():.1f}")
print(f"  Mean R2:   {xgb_results['r2'].mean():.4f}")
print()
best = 'XGBoost' if xgb_results['mape'].mean() < lgb_results['mape'].mean() else 'LightGBM'
print(f'Winner: {best}')